In [196]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [197]:
yerevan_historical = pd.read_csv('../data/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../data/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [198]:
yerevan_historical.drop(columns=[ 'days_in'], inplace=True)

In [199]:
yerevan_actual['backup_status'] = 'not sold'

In [200]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [201]:
df.drop(columns=['links', 'source', 'rent_or_sale'], inplace=True)

In [205]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,sold_date,furniture,renovation,price_per_meter,...,height,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered
79309,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",NaN,1.0,NaN,2314.814815,...,2.8,1.0,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.51312,40.166179


In [203]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [204]:
df = df.dropna(subset=['district']).reset_index(drop=True)

### classifying neighborhoods

In [55]:
df2 = df.head(80589).reset_index(drop=True)

In [32]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

addresses = []

In [33]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error

In [87]:
df['neighborhood'] = addresses

In [63]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)

### small amount of rows that have yerevan classfied replaced with district value 

In [206]:
df = pd.read_csv('../data/yerevan_combined_clean/sale_combined.csv')

In [150]:
for index, row in df[df['neighborhood'] == 'Yerevan'].iterrows():
    df.loc[index, 'neighborhood'] = row['district']

In [151]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [207]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [158]:
# removing neighborhoods with low counts
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [159]:
# removing neighborhoods with low counts
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [166]:
df = pd.concat([sale_historical, sale_actual], axis=0)

for some reason when scraping the data there were missing values for square-meters when price-per-meter had a value and the other way around

In [166]:
for index, row in df.iterrows():
    if np.isnan(row['price_per_meter']):
        continue
    else:
        df.loc[index, 'square_meters'] = row['price'] / row['price_per_meter']

In [167]:
for index, row in df.iterrows():
    if np.isnan(row['square_meters']):
        continue
    else:
        df.loc[index, 'price_per_meter'] = row['price'] / row['square_meters']

In [171]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)

In [14]:
df = pd.read_csv('../data/yerevan_combined_clean/sale_combined.csv')

In [67]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [17]:
sale_actual.drop(columns=[ 'sold_date'], inplace=True)

In [19]:
sale_historical = sale_historical.dropna(subset=['sold_date'])

In [24]:
sale_historical['sold_date'] = pd.to_datetime(sale_historical['sold_date'])

# Extract the date part from the datetime column
sale_historical['sold_date'] = sale_historical['sold_date'].dt.date

In [30]:
sale_historical.to_csv(r'../data/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../data/yerevan_combined_clean/actual_not_sold.csv', index=False)

In [217]:
df = pd.concat([sale_historical,sale_actual], axis=0)

In [219]:
df.drop(columns=[ 'sold_date'], inplace=True)

In [220]:
df_clean = df.dropna().reset_index(drop=True)

In [224]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)
df_clean.to_csv(r'../data/yerevan_combined_clean/sale_clean_combined.csv', index=False)